# La Liga Web Scraping Pipeline
## Project Overview
Before we can predict who wins a football match, we need high-quality, granular data. This project implements an end-to-end web scraping pipeline to extract multi-season match results and detailed shooting statistics for La Liga from FBref.

Rather than just downloading a static dataset, we will be web scraping the data. In this project we will handle recursive page navigation (scraping historical seasons dynamically), merge seperate data sources, manage request throttling to respect host servers, and handles missing data gracefully using exception handling.

Knowing that modern websites actively block automated data collection, in order to do this we will use Selenium. We can think of Selenium as a virtual robot that sits at your computer and browses the internet exactly like a human would. Instead of just sending a raw code request to a server, it physically opens up a real web browser (like Chrome), clicks on links, and lets the page load completely.

The final output being a clean, feature-rich dataset of 1,400+ historical matches saved directly to a CSV, fully optimized for downstream machine learning tasks.

### Scraping our First Page with Requests

In [ ]:
import ssl
# Force Python to ignore unverified SSL certificates so our browser doesn't panic on startup
ssl._create_default_https_context = ssl._create_unverified_context

import undetected_chromedriver as uc
import time

# This is the exact URL we want our browser to visit
standings_url = "https://fbref.com/en/comps/12/La-Liga-Stats"

# Initialize our robotic Chrome browser
options = uc.ChromeOptions()
driver = uc.Chrome(options=options, version_main=150)

# Tell the browser to navigate to the La Liga stats page
driver.get(standings_url)

# Wait 5 seconds.
time.sleep(5) 

# Grab the fully loaded HTML code from the page
html = driver.page_source

### Parsing the HTML Links with BeautifulSoup
In order to parse the HTML we are going to use the BeautifulSoup Library, which is just a popular Python library used for parsing HTML and XML documents to extract, navigate, and modify web data. We will be scraping the data the following way: 

1. **League Standings Parser**:
<br>Download the La Liga standings table, parse the HTML DOM, and use CSS selectors to isolate the specific anchor (< a >) tags belonging to individual squad pages.
2. **Form & Fixture Extractor**:
<br>Navigates to each squad's unique URL and use Pandas' read_html function to convert the Scores & Fixtures table into a DataFrame.
3. **Granular Shooting Stats Scraper**:
<br>Locates the relative link to the team's detailed shooting page, download it, drop multi-level header indexes, and extracts advanced metrics (`Shots`, `Shots on Target`, `Shot Distance`, `FK`, `PK`).


In [ ]:
from bs4 import BeautifulSoup

# Hand the raw HTML off to BeautifulSoup, which acts like a magnifying glass to find specific data tags
soup = BeautifulSoup(html, features="html.parser")

# Search the HTML specifically for our data table
tables = soup.select('table.stats_table')

if len(tables) == 0:
    # If the list is empty, Cloudflare's security still caught us and hid the table
    print("Error: Website Blocked Web Scraping")
else:
    # Extract the actual table object from our BeautifulSoup list
    standings_table = tables[0]
    
    # Find all the links (anchor tags, or 'a') inside that table
    links = standings_table.find_all('a')
    
    # Loop through the links and pull out just the actual URL (the 'href' attribute)
    links = [l.get("href") for l in links]
    
    # Filter the list so we only keep URLs that point to individual squad pages
    links = [l for l in links if l and '/squads/' in l]

    # Formatting team_links to be full urls to use later
    team_links = [f"https://fbref.com{l}" for l in links]

### Extract Match Stats Using Pandas and Requests
In this part of the project, we will extract the match statistics for a specific team (I'm choosing Real Madrid because they're my favorite team) from their team-specific URL. We'll grab the HTML using the requests library and use pandas to instantly parse the match history table into a structured DataFrame. So in order to do complete this phase of the porject we will need to: 

1. **Fetching the Team HTML**:
<br>First, we will define our target team URL and send a GET request to retrieve the page's HTML content.
2. **Locate the Target Table**:
<br>We will inspect the webpage to find our target table. On the team page we are specifically looking for the "Scores & Fixtures" table.
3. **Parse the Table with Pandas**:
<br>Pandas has a function called `pd.read_html()`. This function scans an HTML string for `<table>` tags and automatically converts them into a list of DataFrame objects.
4. **Extract the DataFrame**:
<br>Because `pd.read_html()` always returns a list of DataFrames (even if only one match is found), we need to extract the first element of that list to get our actual table.

In [ ]:
import pandas as pd

team_url = 'https://fbref.com/en/squads/53a2f082/Real-Madrid-Stats'

print(f"Navigating to {team_url}...")
driver.get(team_url)

# Wait 5 seconds.
time.sleep(5) 

# Grab the fully loaded HTML code from the page
team_html = driver.page_source

if 'id="matchlogs_for"' not in team_html:
    print("Error: The table ID 'matchlogs_for' is nowhere in the HTML.")
    if "Cloudflare" in team_html or "Just a moment..." in team_html:
        print("Reason: Cloudflare intercepted the request.")
    else:
        print("Reason: The page loaded, but the layout changed or FBref hid the table.")
else:
    
    # Use the exact HTML ID to extract the table
    from io import StringIO
    matches = pd.read_html(StringIO(team_html), attrs={"id": "matchlogs_for"})
    
    # Pandas returns a list of tables. We want the first one [0].
    real_madrid_df = matches[0]
    
    # Sanity check
    print(real_madrid_df.head())

### Extracting Match Shooting Stats Using Requests and Pandas
While the basic match statistics (win/loss, goals for/against, attendance) are useful, it's not enough for what we need. To build a better machine learning model, we need deeper performance metrics like shots, shots on target, free kicks, and penalty kicks. So in this phase of the project, we will: 

1. Locate the link to the shooting page using Beautiful Soup.
2. Scrape the shooting stats HTML using Requests.
3. Parse the shooting data table into a DataFrame using Pandas.

In [ ]:
soup = BeautifulSoup(team_html, features="html.parser")
links = soup.find_all('a')

# Same logic as before
if len(links) == 0:
    # If the list is empty, Cloudflare's security still caught us and hid the table
    print("Error: Website Blocked Web Scraping")
else:
    links = [l.get("href") for l in links]
    links = [l for l in links if l and 'all_comps/shooting/' in l]

# Navigating to the shooting page
driver.get(f"https://fbref.com{links[0]}")

# Waiting 5 seconds
time.sleep(5)

shooting_html = driver.page_source

driver.quit()

# Add [0] to the end to extract the dataframe from the list
shooting = pd.read_html(StringIO(shooting_html), match="Shooting")[0]

# Sanity Check
print(shooting.head())

### Cleaning and Merging Our Scraped Data with Pandas
Now we have two separate data frames:

1. `real_madrid_df`: The basic match details (date, opponent, result, goals).
2. `shooting`: The detailed shooting stats (shots, shots on target, penalties).

Our goal is to clean up the shooting data and merge it with our basic match data so everything is in one clean, easy-to-use table.

In [ ]:
# Dropping the top level index
shooting.columns = shooting.columns.droplevel()

# merging df's together
real_madrid_data = real_madrid_df.merge(shooting[["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]], on="Date")

# Sanity Check
print(real_madrid_data.head())